# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Research question:** Among FlyRank's tracked content pages, can page-level performance signals (position, CTR, engagement, historical trend) identify which pages are declining and worth a refresh — better than a simple hand-written rule?

**Decision this supports:** Prioritizing a content team's limited refresh capacity — which pages to review first (`snippet_fix`, `content_fix`, `monitor`) — rather than reviewing pages in an arbitrary or purely recency-based order.

In [1]:
research_question = (
    "Can page-level performance signals identify declining content pages "
    "worth a refresh, better than a simple hand-written rule?"
)
decision_supported = (
    "Prioritizing a content team's limited refresh capacity across "
    "snippet_fix / content_fix / monitor actions."
)
print("Question:", research_question)
print("Decision:", decision_supported)

Question: Can page-level performance signals identify declining content pages worth a refresh, better than a simple hand-written rule?
Decision: Prioritizing a content team's limited refresh capacity across snippet_fix / content_fix / monitor actions.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release:** `FlyRank/internship-warehouse` (Hugging Face) — `fact_content_daily_performance`, joined with `dim_content` and `dim_clients`.

**Window used:** March 2026 daily performance, aggregated to one row per (client, page).

**Eligibility filter:** ≥50 GSC impressions OR ≥10 GA4 sessions → **116,512 eligible pages**.

**Excluded:** pages below that threshold (too sparse to score reliably — a single visitor can swing engagement rate from 0% to 100%); raw client/domain identifiers (only `client_hash_id` / `content_hash_id` used throughout, per the public-safe rule).

In [2]:
# same data as the previous week assignments
!pip install -q duckdb huggingface_hub
import duckdb, pandas as pd
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
base = "hf://datasets/FlyRank/internship-warehouse"

df_march = con.sql(f"SELECT * FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/data_0.parquet')").df()

agg_df = df_march.groupby(['client_hash_id', 'content_hash_id']).agg(
    gsc_impressions=('gsc_impressions', 'sum'),
    gsc_clicks=('gsc_clicks', 'sum'),
    gsc_sum_position=('gsc_sum_position', 'sum'),
    ga4_sessions=('ga4_sessions', 'sum'),
    ga4_engaged_sessions=('ga4_engaged_sessions', 'sum'),
).reset_index()

agg_df['gsc_avg_position'] = agg_df['gsc_sum_position'] / agg_df['gsc_impressions']
agg_df.loc[agg_df['gsc_sum_position'] == 0, 'gsc_avg_position'] = pd.NA

eligible_mask = (agg_df['gsc_impressions'] >= 50) | (agg_df['ga4_sessions'] >= 10)
eligible_df = agg_df[eligible_mask].copy()

print(f"Total pages aggregated: {len(agg_df):,}")
print(f"Eligible pages (>=50 impressions OR >=10 sessions): {len(eligible_df):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total pages aggregated: 331,437
Eligible pages (>=50 impressions OR >=10 sessions): 116,512


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Assumptions.** A page's March performance signals are known at the decision point (start of April), and the goal is to rank pages by how likely they are to lose clicks the following month. Scores are decision-support for human review, not proof of cause.

**Label.** `decline_label = 1` if a page's April clicks are below 80% of its March clicks. A page is labeled only if (a) it is still tracked in April and (b) it had at least 5 clicks in March, so the ratio isn't noise from tiny counts.

**Features (all March-only).** `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_sessions`, `ga4_engaged_sessions`, `ctr`, `engagement_rate`, `peer_avg_ctr`, plus position-bucket dummies.

**Baseline.** The Week 4 hand-written rule, scored on the same pages: `snippet_fix` if CTR is below half the position-peer average, else `content_fix` if engagement rate is below 10% (with at least 10 sessions), else `monitor`. Its `rule_score` is the estimated clicks recoverable, used only as the ranking score.

**Models.** Logistic regression (simple, readable) and random forest (can catch interactions).

**Validation design.** 5-fold `GroupKFold` on `client_hash_id`, so no client appears in both train and test, with a zero-overlap assert on every fold. The rule, logistic regression, and random forest are all scored on the same folds. Primary metric is Precision@50, with @10 and @100 as side checks.

**Leakage checks.**
1. Label leakage: neither `decline_label` nor April clicks are in the features.
2. Future leakage: every feature traces back to March-only data.
3. Downstream leakage: the rule's outputs are not used as model inputs.
4. Injection test: a deliberately planted label copy pushes the score toward 1.0, which shows the audit can catch a leak.

Disclosed caveat: `gsc_clicks` and `ctr` are close cousins of the label (both come from March clicks). They are kept because they are known before the decision point, and Section 4 checks how much the model depends on them.

In [3]:
import numpy as np
from sklearn.model_selection import GroupKFold

# April clicks (for the label)
df_april_agg = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS gsc_clicks_apr
    FROM read_parquet('{base}/fact_content_daily_performance/month=2026-04/data_0.parquet')
    GROUP BY client_hash_id, content_hash_id
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [4]:
# Baseline rule (Week 4)
def position_bucket(pos):
    if pd.isna(pos): return 'no_position_data'
    elif pos <= 3:   return '1-3 (top)'
    elif pos <= 10:  return '4-10'
    elif pos <= 20:  return '11-20'
    else:            return '21+'

signal_df = agg_df[agg_df['gsc_impressions'] > 0].copy()
signal_df['ctr'] = signal_df['gsc_clicks'] / signal_df['gsc_impressions']
signal_df['position_bucket'] = signal_df['gsc_avg_position'].apply(position_bucket)
position_avg_ctr = signal_df.groupby('position_bucket')['ctr'].mean().to_dict()

rule_df = agg_df.copy()
rule_df['ctr'] = rule_df['gsc_clicks'] / rule_df['gsc_impressions']
rule_df['engagement_rate'] = rule_df['ga4_engaged_sessions'] / rule_df['ga4_sessions']
rule_df['position_bucket'] = rule_df['gsc_avg_position'].apply(position_bucket)
rule_df['peer_avg_ctr'] = rule_df['position_bucket'].map(position_avg_ctr)
rule_df = rule_df[(rule_df['gsc_impressions'] >= 50) | (rule_df['ga4_sessions'] >= 10)].copy()

In [5]:
conditions = [
    (rule_df['gsc_impressions'] >= 50) & rule_df['ctr'].notna() & rule_df['peer_avg_ctr'].notna()
        & (rule_df['ctr'] < 0.5 * rule_df['peer_avg_ctr']),
    (rule_df['ga4_sessions'] >= 10) & rule_df['engagement_rate'].notna()
        & (rule_df['engagement_rate'] < 0.10),
]
rule_df['action'] = np.select(conditions, ['snippet_fix', 'content_fix'], default='monitor')
rule_df['reason_code'] = np.select(conditions,
    ['CTR_below_half_position_peers', 'engagement_below_10pct_reliable'], default='no_flag_triggered')
rule_df['rule_score'] = np.select(conditions,
    [(rule_df['peer_avg_ctr'] - rule_df['ctr']) * rule_df['gsc_impressions'],
     rule_df['ga4_sessions'] * (0.10 - rule_df['engagement_rate']) * 10], default=0)

In [6]:
# decline label
MIN_MARCH_CLICKS_FOR_LABEL = 5
DECLINE_THRESHOLD = 0.8

merged = rule_df.merge(df_april_agg, on=['client_hash_id', 'content_hash_id'], how='left')
tracked_in_april = merged['gsc_clicks_apr'].notna()
labelable = merged[tracked_in_april & (merged['gsc_clicks'] >= MIN_MARCH_CLICKS_FOR_LABEL)].copy()
labelable['decline_label'] = (labelable['gsc_clicks_apr'] < DECLINE_THRESHOLD * labelable['gsc_clicks']).astype(int)

print(f"Eligible pages:            {len(rule_df):,}")
print(f"Not tracked in April:      {(~tracked_in_april).sum():,}")
print(f"Labelable pages:           {len(labelable):,}")
print(f"Base rate (decline == 1):  {labelable['decline_label'].mean():.3f}")

Eligible pages:            116,512
Not tracked in April:      0
Labelable pages:           28,805
Base rate (decline == 1):  0.545


In [7]:
# Features (march-only)
FEATURES = ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions',
            'ga4_engaged_sessions', 'ctr', 'engagement_rate', 'peer_avg_ctr']

def build_X(df, feature_cols):
    X = df[feature_cols].astype(float).replace([np.inf, -np.inf], np.nan)
    if 'gsc_avg_position' in X.columns:
        worst = X['gsc_avg_position'].max()
        X['gsc_avg_position'] = X['gsc_avg_position'].fillna(worst + 10 if pd.notna(worst) else 100)
    for c in ['ga4_sessions', 'ga4_engaged_sessions', 'engagement_rate', 'ctr']:
        if c in X.columns:
            X[c] = X[c].fillna(0)
    if 'peer_avg_ctr' in X.columns:
        X['peer_avg_ctr'] = X['peer_avg_ctr'].fillna(X['peer_avg_ctr'].median())
    return X.join(pd.get_dummies(df['position_bucket'], prefix='pos', drop_first=True))

def precision_at_k(ids, scores, labels, k):
    temp = pd.DataFrame({'content_hash_id': ids, 'score': scores, 'label': labels})
    ranked = temp.sort_values(['score', 'content_hash_id'], ascending=[False, True])
    return ranked.head(k)['label'].mean()

X_full = build_X(labelable, FEATURES)
X_full = X_full.astype(float)
assert not X_full.isna().any().any(), "NaNs left in X_full"
y = labelable['decline_label'].values
groups = labelable['client_hash_id'].values
gkf = GroupKFold(n_splits=5)

In [8]:
# Leakage checks
assert 'decline_label' not in FEATURES and 'gsc_clicks_apr' not in FEATURES, "label / next-month leak"
allowed = set(FEATURES) | {c for c in X_full.columns if c.startswith('pos_')}
assert set(X_full.columns) <= allowed, "unexpected column in X_full"
assert not ({'action', 'reason_code', 'rule_score'} & set(FEATURES)), "rule output used as a feature"
for train, test in gkf.split(X_full, y, groups):
    assert len(set(groups[train]) & set(groups[test])) == 0, "client leakage across a fold"
print("Leakage checks passed. Features:", list(X_full.columns))

Leakage checks passed. Features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions', 'ctr', 'engagement_rate', 'peer_avg_ctr', 'pos_11-20', 'pos_21+', 'pos_4-10']


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*


All three methods are scored on the same 28,805 labelable pages and the same 5 client-grouped folds (no client appears in both train and test). Primary metric is Precision@50: of the 50 pages each method ranks highest in a fold, what share actually declined? Averages across folds:

| Method | Precision@10 | Precision@50 | Precision@100 |
|---|---|---|---|
| Rule (baseline) | 0.480 | 0.472 | 0.498 |
| Logistic regression | 0.580 | **0.592** | 0.574 |
| Random forest | 0.600 | 0.560 | 0.566 |
| *Random pick (base rate)* | *0.545* | *0.545* | *0.545* |

**What was measured.**
- The rule's top-ranked pages declined *less often* than a random pick (0.472 vs 0.545). At P@50, the rule's ranking did not beat chance in this data.
- Logistic regression's P@50 was about 12 points above the rule and about 5 points above a random pick. It beat the rule in 4 of 5 held-out-client folds.
- Random forest also beat the rule in 4 of 5 folds, but its P@50 (0.560) was slightly below logistic regression's (0.592). Random forest was ahead only at P@10 (0.60 vs 0.58).
- Both models did worse than the rule on the same fold (fold 4). Random forest fell below the base rate there (0.26 vs 0.546), so the learned pattern did not transfer to that group of clients.

**Read.** Under a client-grouped split, the models show a directional improvement over the rule, and the size of it varies a lot by fold (logistic regression P@50 ranged from 0.42 to 0.78). With 5 folds and a single run each, this is decision-support evidence, not a precise performance estimate. Logistic regression is the recommended model: it scored at least as well as the forest at P@50 and P@100, and it is easier to explain.

**Sensitivity check.** `gsc_clicks` and `ctr` are close relatives of the label (both come from March clicks). With the same random seed on both runs, removing them changed random forest P@50 from 0.560 to 0.556, a negligible difference. In this run the model does not appear to depend on those two features. An earlier check in a previous notebook showed a larger drop (0.572 to 0.520), but it compared runs with different random seeds, so I treat it as inconclusive.

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

k_values = [10, 50, 100]
rule_scores_all = labelable['rule_score'].values
ids_all = labelable['content_hash_id'].values

In [10]:
# Rule vs logistic regression vs random forest
results = []
for fold, (train, test) in enumerate(gkf.split(X_full, y, groups)):
    scaler = StandardScaler()
    Xtrain = scaler.fit_transform(X_full.iloc[train])
    Xtest = scaler.transform(X_full.iloc[test])
    logreg = LogisticRegression(max_iter=1000)
    logreg.fit(Xtrain, y[train])
    lr_preds = logreg.predict_proba(Xtest)[:, 1]

    rf = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=20, random_state=0)
    rf.fit(X_full.iloc[train], y[train])
    rf_preds = rf.predict_proba(X_full.iloc[test])[:, 1]

    row = {'fold': fold, 'n_test': len(test), 'base_rate': round(y[test].mean(), 3)}
    for k in k_values:
        row[f'rule_p{k}']   = precision_at_k(ids_all[test], rule_scores_all[test], y[test], k)
        row[f'logreg_p{k}'] = precision_at_k(ids_all[test], lr_preds, y[test], k)
        row[f'rf_p{k}']     = precision_at_k(ids_all[test], rf_preds, y[test], k)
    results.append(row)

results_df = pd.DataFrame(results)
print("Per-fold Precision@50:")
print(results_df[['fold', 'n_test', 'base_rate', 'rule_p50', 'logreg_p50', 'rf_p50']].round(3).to_string(index=False))

Per-fold Precision@50:
 fold  n_test  base_rate  rule_p50  logreg_p50  rf_p50
    0    6799      0.514      0.40        0.68    0.74
    1    5502      0.733      0.66        0.78    0.80
    2    5502      0.483      0.36        0.52    0.56
    3    5502      0.455      0.36        0.56    0.44
    4    5500      0.546      0.58        0.42    0.26


In [11]:
summary = pd.DataFrame({
    'Precision@10':  [results_df[f'{m}_p10'].mean()  for m in ['rule', 'logreg', 'rf']],
    'Precision@50':  [results_df[f'{m}_p50'].mean()  for m in ['rule', 'logreg', 'rf']],
    'Precision@100': [results_df[f'{m}_p100'].mean() for m in ['rule', 'logreg', 'rf']],
}, index=['Rule (baseline)', 'Logistic regression', 'Random forest']).round(3)
print("\nAverages across 5 client-held-out folds:")
print(summary)
print(f"\nOverall base rate (random-pick precision): {y.mean():.3f}")

for name, col in [('Logistic regression', 'logreg_p50'), ('Random forest', 'rf_p50')]:
    print(f"{name} vs rule at P@50: better in {(results_df[col] > results_df['rule_p50']).sum()}/5 folds, "
          f"worse in {(results_df[col] < results_df['rule_p50']).sum()}/5")


Averages across 5 client-held-out folds:
                     Precision@10  Precision@50  Precision@100
Rule (baseline)              0.48         0.472          0.498
Logistic regression          0.58         0.592          0.574
Random forest                0.60         0.560          0.566

Overall base rate (random-pick precision): 0.545
Logistic regression vs rule at P@50: better in 4/5 folds, worse in 1/5
Random forest vs rule at P@50: better in 4/5 folds, worse in 1/5


In [12]:
# Sensitivity: remove clicks/ctr (close cousins of the label)
FEATURES_NO_CLICKS = [f for f in FEATURES if f not in ['gsc_clicks', 'ctr']]
X_nc = build_X(labelable, FEATURES_NO_CLICKS)
sens = []
for fold, (train, test) in enumerate(gkf.split(X_nc, y, groups)):
    rf = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=20, random_state=0)
    rf.fit(X_nc.iloc[train], y[train])
    s = rf.predict_proba(X_nc.iloc[test])[:, 1]
    sens.append(precision_at_k(ids_all[test], s, y[test], 50))
print(f"\nRF Precision@50 with clicks/CTR:    {results_df['rf_p50'].mean():.3f}")
print(f"RF Precision@50 without clicks/CTR: {np.mean(sens):.3f}")


RF Precision@50 with clicks/CTR:    0.560
RF Precision@50 without clicks/CTR: 0.556


## 5. Limitations

*What this work cannot claim.*

- **Decision-support, not causation.** A high score means a page matches a pattern that preceded click declines in this data. It does not show that anything is broken or that refreshing the page would recover clicks.
- **The label is a click ratio, not a diagnosis.** A page "declines" if April clicks are below 80% of March clicks. That includes seasonal dips, ranking noise, and demand changes that no refresh could fix. One month pair (March to April) is the only outcome tested here.
- **Only pages with real click volume are covered.** About a quarter of eligible pages had 5 or more March clicks and could be labeled. The results say nothing about low-traffic pages.
- **The baseline ranked below chance.** The rule's top 50 declined less often (0.472) than a random pick (0.545). Its score mixes two scales (estimated clicks for `snippet_fix`, weighted sessions for `content_fix`), with an arbitrary `× 10` weight. So "the model beat the rule" partly means the rule's ranking was weak. It does not show the model is strong.
- **Results vary a lot by fold.** Logistic regression P@50 ranged from 0.42 to 0.78 across 5 client-held-out folds. On the weakest fold both models scored below the rule and the random forest scored below the base rate. Each estimate comes from a single run, so the numbers are a rough band, not a precise measurement. Earlier runs of the same code on a different data pull gave noticeably different validated numbers, which supports this.
- **Missing engagement was filled with 0.** About a quarter of labeled pages had no GA4 sessions, and their engagement rate was set to 0. That treats "no traffic data" the same as "visitors came and none engaged."
- **Both models miss some declines.** A strong page in March (top-3 position, high traffic) gives the model little warning, and pages that are chronically weak but stable can be flagged as declining. These blind spots were seen in the earlier error analysis.
- **Features are close to the label.** `gsc_clicks` and `ctr` come from the same March clicks the label is built from. They are known before the decision point, but they mean part of the signal may just be "pages with few clicks can lose a large share of them."

In [13]:
n_eligible = len(rule_df)
n_labelable = len(labelable)
zero_sessions = (labelable['ga4_sessions'] == 0).mean()
fold_spread = results_df[['rule_p50', 'logreg_p50', 'rf_p50']].agg(['min', 'max']).round(3)

print(f"Labelable pages: {n_labelable:,} of {n_eligible:,} eligible ({n_labelable / n_eligible:.1%})")
print(f"Base rate (random pick precision): {y.mean():.3f}")
print(f"Rule P@50 vs base rate: {results_df['rule_p50'].mean():.3f} vs {y.mean():.3f}")
print(f"Labeled pages with zero GA4 sessions (engagement_rate filled with 0): {zero_sessions:.1%}")
print("\nP@50 spread across folds:")
print(fold_spread)
weakest = results_df.loc[results_df['logreg_p50'].idxmin(), ['fold', 'rule_p50', 'logreg_p50', 'rf_p50']]
print("\nWeakest fold for logistic regression:")
print(weakest.round(3).to_string())

Labelable pages: 28,805 of 116,512 eligible (24.7%)
Base rate (random pick precision): 0.545
Rule P@50 vs base rate: 0.472 vs 0.545
Labeled pages with zero GA4 sessions (engagement_rate filled with 0): 24.6%

P@50 spread across folds:
     rule_p50  logreg_p50  rf_p50
min      0.36        0.42    0.26
max      0.66        0.78    0.80

Weakest fold for logistic regression:
fold          4.00
rule_p50      0.58
logreg_p50    0.42
rf_p50        0.26


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.